# Shared parking depth: full-pool adapter-only pilot
**PARKING-SHARED-ADAPTER-FULLPOOL-2026-10-27-r1**

Question: can the small camera adapter improve on the original depth head when trained on the entire existing training pool? Only **1,553 adapter parameters** learn. The reviewed epoch-7 depth head, all BatchNorm state, Urban body, detector and projected corners stay frozen. This is supervised depth training, **not knowledge distillation**.

The 16-image sanity check remains failed on its original mixed-loss criterion; its saved endpoint did not improve overall development error. This is a separately approved exploratory pilot, not a claim that the earlier gate passed. Both arms restart from epoch 7 plus a fresh identical zero-output adapter, not the tiny overfit endpoint.

Two matched arms: fixed reference camera versus actual calibrated camera. Five epochs per arm, batch 8, AdamW LR 0.001, weight decay 0, Huber delta 0.2 m, unchanged 0.8 near / 0.2 broad objective. Each epoch visits all 1,536 training images once in the broad stream, plus four critical-pool draws per batch: **384 updates per epoch, 1,920 per arm**. This doubles the old joint-camera pilot's updates per epoch; it is not a compute-matched comparison with that old experiment.

Run sections 1–2 on CPU first. A GPU may be selected after CPU staging; reconnecting clears local files, so rerun from section 1. Do not alter completed runs or receipts. A changed GPU/package runtime requires a new RUN_ID. No held-out evaluation, scale fitting, export, extra epochs, or iPhone change is started by this workflow.

## 1. Mount Drive and update main (CPU)
The dataset ZIP and warm-start result ZIP already exist in Drive. No new dataset upload is needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import collections, datetime, hashlib, json, os, shlex, subprocess, sys
REVISION = 'PARKING-SHARED-ADAPTER-FULLPOOL-2026-10-27-r1'
RUN_ID = 'shared_adapter_frozen_fullpool_r1'
MOBILE = Path('/content/mobile_adas3d')
BASE = Path('/content/drive/MyDrive')
PACKAGE = BASE/'mobile_adas3d_inputs/parking_shared_near/trainval_inputs_r2'
ARCHIVE = PACKAGE/'shared_near_inputs.zip'
RECEIPT = PACKAGE/'archive_receipt.json'
WARMSTART = BASE/'mobile_adas3d_outputs/parking_shared_near/shared_depth_near_trainval_r1/parking_shared_near_results.zip'
INPUTS = Path('/content/shared_near_inputs_shared_depth_near_trainval_r1')
CACHE = Path('/content')/('shared_adapter_features_'+RUN_ID)
REUSE_CACHES = [Path('/content/shared_camera_features_shared_camera_5epoch_r1'), Path('/content/shared_near_features_shared_depth_near_trainval_r1')]
SOURCE = Path('/content/Urban_Shared_Near_Source')
WEIGHTS = Path('/content/urban_shared_near_weights')
OUTPUT = BASE/'mobile_adas3d_outputs/parking_shared_adapter'/RUN_ID
DEPS = Path('/content')/('shared_adapter_deps_'+RUN_ID)
PYTHON = sys.executable
SCRIPT = MOBILE/'scripts/train_parking_shared_adapter.py'
OUTPUT.mkdir(parents=True, exist_ok=True)
def run(command, name, cwd=None):
    command = [str(x) for x in command]
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    log = OUTPUT/'colab_logs'/f'{name}_{stamp}.log'
    log.parent.mkdir(parents=True, exist_ok=True)
    print('+', shlex.join(command), '\nDurable log:', log, flush=True)
    env = os.environ.copy()
    env['PYTHONPATH'] = str(DEPS)+os.pathsep+str(MOBILE)
    env['YOLO_AUTOINSTALL'] = 'false'
    tail = collections.deque(maxlen=25)
    with log.open('w') as stream:
        process = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True); stream.write(line); stream.flush(); tail.append(line.rstrip())
        finally:
            process.stdout.close()
        code = process.wait(); stream.write(f'\nExit code: {code}\n')
    if code: raise RuntimeError(f'Exit {code}; preserve run and inspect {log}\n'+'\n'.join(tail))
def stage(name):
    command = [PYTHON, '-X', 'faulthandler', '-u', SCRIPT, name, '--inputs', INPUTS, '--warmstart', WARMSTART, '--source', SOURCE, '--weights', WEIGHTS, '--cache', CACHE, '--output', OUTPUT, '--device', 'cuda']
    for candidate in REUSE_CACHES: command += ['--reuse-cache', candidate]
    run(command, name, MOBILE)
url = '/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not MOBILE.exists(): run(['git','clone',url,MOBILE], 'clone')
branch = subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE,text=True).strip()
if branch != 'main': raise RuntimeError(f'Use main, not {branch!r}; preserve other branches')
run(['git','pull','--ff-only'], 'update_main', MOBILE)
for name in ['train_parking_shared_adapter.py', 'sanity_parking_shared_adapter.py']:
    if not (MOBILE/'scripts'/name).is_file(): raise FileNotFoundError(MOBILE/'scripts'/name)
print(REVISION, '\nOutput:', OUTPUT)


## 2. Verify and stage on CPU
Four copy workers reuse verified local files and show progress. No GPU or model is needed for this section. Correct WARMSTART above if your completed ZIP is elsewhere.

In [ ]:
for path in [ARCHIVE, RECEIPT, WARMSTART]:
    if not path.is_file(): raise FileNotFoundError(f'CPU input missing: {path}. Do not activate GPU yet.')
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        while block := stream.read(1024*1024): h.update(block)
    return h.hexdigest()
receipt = json.loads(RECEIPT.read_text())
expected = 'ec3062ce299c7ac70e537e640b16950615bd0d08ed601c823decc28d2d87d1cb'
if receipt['sha256'] != expected or digest(ARCHIVE) != expected: raise RuntimeError('Input archive changed; preserve it')
run([PYTHON, MOBILE/'scripts/parking_shared_near_data.py', 'restore', '--archive', ARCHIVE, '--output', INPUTS], 'restore_inputs', MOBILE)
stage('prepare')
prepared = json.loads((OUTPUT/'prepared_inputs.json').read_text())
assert prepared['budget']['updates_per_arm']==1920 and prepared['budget']['unique_broad_images_per_epoch']==1536
print('CPU preparation passed. Select a GPU runtime, then rerun sections 1–2 after reconnecting.')


## 3. Dependencies, tests and GPU smoke
Keep the current runtime's Torch/CUDA. No old CUDA toolkit, Core ML, or global package replacement. Only the pinned Ultralytics and THOP go into the isolated task directory. Smoke performs image forward and ground-truth backward with **zero optimizer updates**.

In [ ]:
run([PYTHON,'-m','pip','install','--quiet','--no-deps','--upgrade','--target',DEPS,'ultralytics==8.4.61','ultralytics-thop==2.0.18'], 'task_dependencies', MOBILE)
run([PYTHON,'-c',"import torch,cv2,numpy,PIL,scipy; from importlib.metadata import version; assert version('ultralytics')=='8.4.61'; assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.__version__,torch.cuda.get_device_name(0)); print(torch.ones(8,device='cuda').sum().item())"], 'cuda_preflight', MOBILE)
for pattern in ['test_parking_shared_adapter_sanity.py', 'test_parking_shared_adapter_pilot.py']:
    run([PYTHON,'-m','unittest','discover','-s','tests','-p',pattern,'-v'], 'tests_'+pattern, MOBILE)
stage('smoke')
smoke = json.loads((OUTPUT/'runtime_smoke.json').read_text())
assert smoke['complete'] and smoke['both_arms_bit_exact_epoch7_parity'] and smoke['optimizer_steps']==0


## 4. Reuse or cache frozen features once
Old caches are read-only. They are reused only if runtime, data, producer and body identities match and all 1,792 samples verify. Otherwise this run gets its own local cache. Resume reruns verify existing features rather than recopying images.

In [ ]:
stage('cache')


## 5. Start the fixed-budget training pilot
**This cell starts training.** Two arms × five epochs × 384 updates = 3,840 optimizer updates total. Both arms have the same initial state, batches, loss and budget. Only the calibration input differs. Original depth weights/BN and the detector stay frozen.

Checkpoint selection includes epoch zero. An interrupted epoch restarts from the last completed checkpoint; rerun the same notebook with the same runtime/RUN_ID. No automatic budget extension. Development data have already been inspected and are not a fresh blind test.

In [ ]:
stage('train')
result = json.loads((OUTPUT/'adapter_pilot_results.json').read_text())
assert result['complete'] and result['frozen_body_depth_bn_detection_unchanged']
assert not result['prior_tiny_sanity_all_checks_passed'] and not result['heldout_evaluated']
assert not result['phone_app_changed'] and not result['deployment_qualified']
print(json.dumps(result, indent=2))


## 6. Package and return results
Prospective research checks: ≥5% overall development improvement; each class/range cell no more than 2 cm worse; broad error no more than 10 cm worse; Vehicle 2–4 m beats the fixed-4 m diagnostic. Separately test ≥2% improvement over the parameter-matched fixed-camera control. These are descriptive experiment checks, **not** the product's ±10 cm acceptance threshold.

The ZIP contains initialization, selected and final checkpoints for both arms—even if epoch zero wins—plus histories, provenance and code snapshots. Return it before any extra training, held-out evaluation, export or phone replacement.

In [ ]:
stage('bundle')
BUNDLE = OUTPUT/'parking_shared_adapter_results.zip'
print('Return this file:', BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
